# Phase 06 — Basic dense retrieval

This notebook implements the first retrieval path:

```text
Business question → BAAI/bge-m3 dense embedding → in-memory Qdrant cosine search → Top-K chunks
```

It evaluates several Top-K values, displays provenance-rich results, and provides a manual relevance review worksheet. The notebook indexes only the **real** BGE-M3 artifact from Phase 04; it never substitutes a model, mock vector, score, or result. Because Phase 05 uses Qdrant in-memory mode, this notebook reconstructs the same ephemeral collection from the reusable embedding artifact before searching.

> **Precondition:** This notebook can run retrieval only after Phase 04 successfully creates `data/processed/introduction_to_business_bge_m3_embeddings.jsonl` using the exact `BAAI/bge-m3` model.

In [1]:
from __future__ import annotations

import json
import uuid
from pathlib import Path
from typing import Any

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
EMBEDDING_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_bge_m3_embedding_status.json"
EMBEDDINGS_PATH = PROCESSED_DIR / "introduction_to_business_bge_m3_embeddings.jsonl"
QDRANT_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_qdrant_indexing_status.json"
RETRIEVAL_STATUS_PATH = PROCESSED_DIR / "introduction_to_business_basic_retrieval_status.json"
RETRIEVAL_RESULTS_PATH = PROCESSED_DIR / "introduction_to_business_basic_retrieval_results.json"

MODEL_NAME = "BAAI/bge-m3"
COLLECTION_NAME = "openstax_introduction_to_business_bge_m3_retrieval"
TOP_K_VALUES = (3, 5, 8)
REQUIRED_PAYLOAD_FIELDS = ("chunk_id", "text", "source", "page", "chapter", "section")

def write_json(path: Path, payload: dict[str, Any] | list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

print(f"Project root: {PROJECT_ROOT}")
print(f"Embedding artifact expected at: {EMBEDDINGS_PATH}")

Project root: /home/ubuntu/business-knowledge-ai
Embedding artifact expected at: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_bge_m3_embeddings.jsonl


## Sample business questions and manual review criteria

The questions span foundational business concepts, ownership, planning, marketing, and finance. They are **test queries**, not generated answers. For each configuration, manually check whether the retrieved chunks are on-topic, contain adequate supporting context, carry usable page/chapter/section citations, and avoid redundant near-duplicates.

In [2]:
SAMPLE_QUESTIONS = [
    {
        "query_id": "business-foundations",
        "question": "What role do businesses play in an economy?",
        "manual_review_focus": "Definitions of business, value creation, and economic contribution.",
    },
    {
        "query_id": "ownership-choice",
        "question": "What factors should an entrepreneur consider when choosing a business ownership structure?",
        "manual_review_focus": "Ownership forms, liability, control, taxation, and funding trade-offs.",
    },
    {
        "query_id": "management-planning",
        "question": "How does the planning process help managers set organizational goals?",
        "manual_review_focus": "Planning stages, objectives, and managerial decision-making.",
    },
    {
        "query_id": "marketing-selling",
        "question": "How is marketing different from selling?",
        "manual_review_focus": "Marketing concept, customer value, and the role of selling.",
    },
    {
        "query_id": "financial-statements",
        "question": "How do financial statements help business owners make decisions?",
        "manual_review_focus": "Income statement, balance sheet, cash flow, and business decisions.",
    },
]

MANUAL_REVIEW_CHECKLIST = (
    "Assess topical relevance, coverage of the stated focus, citation metadata, and redundancy. "
    "Record which K value gives the clearest useful context without excess noise."
)

for item in SAMPLE_QUESTIONS:
    print(f"[{item['query_id']}] {item['question']}")
print(f"Top-K configurations: {TOP_K_VALUES}")

[business-foundations] What role do businesses play in an economy?
[ownership-choice] What factors should an entrepreneur consider when choosing a business ownership structure?
[management-planning] How does the planning process help managers set organizational goals?
[marketing-selling] How is marketing different from selling?
[financial-statements] How do financial statements help business owners make decisions?
Top-K configurations: (3, 5, 8)


In [3]:
embedding_status = json.loads(EMBEDDING_STATUS_PATH.read_text(encoding="utf-8"))
qdrant_status = json.loads(QDRANT_STATUS_PATH.read_text(encoding="utf-8"))
embeddings_generated = bool(embedding_status.get("embeddings_generated"))
embedding_artifact_exists = EMBEDDINGS_PATH.is_file() and EMBEDDINGS_PATH.stat().st_size > 0
PIPELINE_READY = embeddings_generated and embedding_artifact_exists

preflight = {
    "phase": "06_basic_retrieval",
    "query_embedding_model": MODEL_NAME,
    "embedding_generation_status": embedding_status.get("status"),
    "qdrant_indexing_status": qdrant_status.get("status"),
    "embedding_artifact_exists": embedding_artifact_exists,
    "embeddings_generated": embeddings_generated,
    "top_k_values": list(TOP_K_VALUES),
    "no_model_substitution": True,
    "no_fabricated_embeddings_scores_or_results": True,
    "bm25_implemented": False,
    "hybrid_retrieval_implemented": False,
    "reranking_implemented": False,
    "llm_implemented": False,
    "langgraph_implemented": False,
}

if not PIPELINE_READY:
    preflight.update({
        "status": "blocked_missing_real_bge_m3_embeddings",
        "limitation": (
            "Phase 04 did not produce a real BAAI/bge-m3 embedding artifact. "
            "Query embedding, Qdrant search, scores, and Top-K results are intentionally not generated."
        ),
    })
    write_json(RETRIEVAL_STATUS_PATH, preflight)
    print("Basic retrieval blocked: real BGE-M3 embeddings are unavailable.")
    print(f"Phase 04 status: {embedding_status.get('status')}")
else:
    preflight["status"] = "ready_for_real_basic_retrieval"
    print("Preflight passed: real BGE-M3 artifact found.")

Basic retrieval blocked: real BGE-M3 embeddings are unavailable.
Phase 04 status: blocked_by_resource_preflight


## Real retrieval path

When preflight passes, the next cells load the reusable real-vector artifact, reconstruct an in-memory cosine Qdrant collection, load the exact `BAAI/bge-m3` model for the query side, L2-normalize query vectors, and search the collection for every Top-K setting. No lexical, hybrid, reranking, or answer-generation stage is included.

In [4]:
if PIPELINE_READY:
    import numpy as np
    from qdrant_client import QdrantClient, models

    embedding_records = [
        json.loads(line)
        for line in EMBEDDINGS_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if not embedding_records:
        raise ValueError("Embedding artifact is empty; retrieval is intentionally stopped.")
    if any(any(field not in record for field in REQUIRED_PAYLOAD_FIELDS) for record in embedding_records):
        raise ValueError("Embedding artifact lacks required citation payload fields.")

    def vector_from_record(record: dict[str, Any]) -> list[float]:
        vector = record.get("embedding", record.get("dense_embedding"))
        if not isinstance(vector, list) or not vector:
            raise ValueError("A record lacks a real dense embedding.")
        return [float(value) for value in vector]

    vector_dimension = len(vector_from_record(embedding_records[0]))
    if vector_dimension != 1024:
        raise ValueError(f"Expected BGE-M3 dimension 1024, got {vector_dimension}.")
    if any(len(vector_from_record(record)) != vector_dimension for record in embedding_records):
        raise ValueError("Embedding dimensions are inconsistent.")

    client = QdrantClient(":memory:")
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=vector_dimension, distance=models.Distance.COSINE),
    )
    points = [
        models.PointStruct(
            id=str(uuid.uuid5(uuid.NAMESPACE_URL, record["chunk_id"])),
            vector=vector_from_record(record),
            payload={field: record[field] for field in REQUIRED_PAYLOAD_FIELDS},
        )
        for record in embedding_records
    ]
    client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
    assert client.count(COLLECTION_NAME, exact=True).count == len(points)
    print(f"In-memory Qdrant points: {len(points):,}; dimension: {vector_dimension}; distance: cosine")
else:
    embedding_records: list[dict[str, Any]] = []
    print("In-memory collection not created because preflight is blocked.")

In-memory collection not created because preflight is blocked.


In [5]:
if PIPELINE_READY:
    from FlagEmbedding import BGEM3FlagModel

    query_model = BGEM3FlagModel(MODEL_NAME, use_fp16=True)
    query_texts = [item["question"] for item in SAMPLE_QUESTIONS]
    encoded = query_model.encode(query_texts, batch_size=4, max_length=8192)
    raw_query_vectors = np.asarray(encoded["dense_vecs"], dtype=np.float32)
    norms = np.linalg.norm(raw_query_vectors, axis=1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("A query produced a zero vector; search is intentionally stopped.")
    normalized_query_vectors = raw_query_vectors / norms
    assert normalized_query_vectors.shape == (len(SAMPLE_QUESTIONS), 1024)

    retrieval_runs: list[dict[str, Any]] = []
    for question, query_vector in zip(SAMPLE_QUESTIONS, normalized_query_vectors, strict=True):
        for top_k in TOP_K_VALUES:
            response = client.query_points(
                collection_name=COLLECTION_NAME,
                query=query_vector.tolist(),
                limit=top_k,
                with_payload=True,
                with_vectors=False,
            )
            results = [
                {
                    "rank": rank,
                    "score": float(point.score),
                    "text": point.payload["text"],
                    "page": point.payload["page"],
                    "chapter": point.payload["chapter"],
                    "section": point.payload["section"],
                    "source": point.payload["source"],
                    "chunk_id": point.payload["chunk_id"],
                }
                for rank, point in enumerate(response.points, start=1)
            ]
            assert len(results) == top_k
            retrieval_runs.append({
                "query_id": question["query_id"],
                "question": question["question"],
                "top_k": top_k,
                "manual_review_focus": question["manual_review_focus"],
                "results": results,
            })
    print(f"Executed {len(retrieval_runs)} query/K retrieval runs.")
else:
    retrieval_runs: list[dict[str, Any]] = []
    print("Query embedding and search skipped: no real BGE-M3 model/artifact may be used.")

Query embedding and search skipped: no real BGE-M3 model/artifact may be used.


In [6]:
if PIPELINE_READY:
    for run in retrieval_runs:
        print("=" * 100)
        print(f"Question: {run['question']} | Top-K: {run['top_k']}")
        print(f"Manual review focus: {run['manual_review_focus']}")
        for result in run["results"]:
            preview = result["text"].replace("\n", " ")[:500]
            print(
                f"#{result['rank']} | score={result['score']:.4f} | page={result['page']} | "
                f"chapter={result['chapter']} | section={result['section']}\n"
                f"source={result['source']}\n{preview}\n"
            )

    review_template = [
        {
            "query_id": item["query_id"],
            "top_k": top_k,
            "relevance_pass": None,
            "coverage_pass": None,
            "citation_metadata_pass": None,
            "redundancy_note": "",
            "reviewer_note": "",
        }
        for item in SAMPLE_QUESTIONS
        for top_k in TOP_K_VALUES
    ]
    print("Manual review checklist:", MANUAL_REVIEW_CHECKLIST)
    print(json.dumps(review_template[:3], indent=2))
else:
    print("No scores, chunks, or retrieval-quality observations were produced; the preflight prevented fabrication.")

No scores, chunks, or retrieval-quality observations were produced; the preflight prevented fabrication.


In [7]:
if PIPELINE_READY:
    write_json(RETRIEVAL_RESULTS_PATH, retrieval_runs)
    final_status = {
        **preflight,
        "status": "completed_with_real_bge_m3_qdrant_retrieval",
        "question_count": len(SAMPLE_QUESTIONS),
        "top_k_values": list(TOP_K_VALUES),
        "query_vector_dimension": 1024,
        "qdrant_distance": "cosine",
        "result_artifact": str(RETRIEVAL_RESULTS_PATH.relative_to(PROJECT_ROOT)),
        "manual_review_required": True,
    }
    write_json(RETRIEVAL_STATUS_PATH, final_status)
    print(f"Saved real retrieval results: {RETRIEVAL_RESULTS_PATH}")
else:
    print(f"Saved truthful preflight status: {RETRIEVAL_STATUS_PATH}")

Saved truthful preflight status: /home/ubuntu/business-knowledge-ai/data/processed/introduction_to_business_basic_retrieval_status.json


## Phase boundary

Phase 06 is limited to dense BGE-M3 query embedding and basic Qdrant Top-K verification. It does **not** implement BM25, sparse or hybrid retrieval, fusion, reranking, LLM calls, answer generation, or LangGraph.